# Autoencoder HIDS — Exploratory Data Analysis
**Project:** Autoencoder HIDS for Encrypted Network Traffic — Flow-Based Behavioral Features  
**Authors:** Nelson Mivule (MAY23/CSF/2990U/F) & Erecho Jonathan (MAY23/CSF/3335U/F)  
**Supervisor:** Mr. Senfuka Huzaifah

In [ ]:
import os, gc, glob, warnings
import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker  as mticker
import seaborn as sns
from scipy                  import stats
from sklearn.preprocessing  import MinMaxScaler
from sklearn.decomposition  import PCA
from sklearn.ensemble       import RandomForestClassifier

warnings.filterwarnings("ignore")

BRAND  = "#1A237E"   # deep indigo  — titles, axes labels
ACCENT = "#E53935"   # vivid red    — malicious / anomaly
BLUE   = "#1E88E5"   # mid blue     — benign
GREY   = "#546E7A"   # blue-grey    — secondary text / grid
BG     = "#F7F9FC"   # near-white   — figure background
CARD   = "#FFFFFF"   # axes background
GRID   = "#DDE3EE"   # subtle grid lines
PAL    = {"Benign": BLUE, "Malicious": ACCENT}

plt.rcParams.update({
    # Figure
    "figure.dpi"          : 150,
    "figure.facecolor"    : BG,
    "figure.edgecolor"    : BG,
    "savefig.facecolor"   : BG,
    "savefig.edgecolor"   : BG,
    # Axes
    "axes.facecolor"      : CARD,
    "axes.edgecolor"      : GREY,
    "axes.linewidth"      : 0.8,
    "axes.grid"           : True,
    "axes.spines.top"     : False,
    "axes.spines.right"   : False,
    "axes.labelcolor"     : BRAND,
    "axes.titlecolor"     : BRAND,
    "axes.titlesize"      : 11,
    "axes.titleweight"    : "bold",
    "axes.labelsize"      : 9,
    # Grid
    "grid.color"          : GRID,
    "grid.linewidth"      : 0.6,
    "grid.alpha"          : 0.9,
    # Ticks
    "xtick.color"         : GREY,
    "ytick.color"         : GREY,
    "xtick.labelsize"     : 8,
    "ytick.labelsize"     : 8,
    # Legend
    "legend.framealpha"   : 0.92,
    "legend.edgecolor"    : GRID,
    "legend.fontsize"     : 8,
    # Font
    "font.family"         : "DejaVu Sans",
    "font.size"           : 9,
    "text.color"          : BRAND,
})

def _watermark(fig, text="UTAMU  ·  AUTOENCODER HIDS"):
    fig.text(0.99, 0.005, text, ha="right", va="bottom",
             fontsize=6, color=GREY, alpha=0.6, fontstyle="italic")

def save_fig(fig, name):
    _watermark(fig)
    path = os.path.join(FIGURES_PATH, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved → {path}")

# Paths 
BASE_DIR       = "/home/l391lninj1/Desktop/AUTOENCODER_HIDS"
DATASET_PATH   = os.path.join(BASE_DIR, "experiments", "datasets", "CIC-IDS2017")
TEST_DATA_PATH = os.path.join(BASE_DIR, "experiments", "datasets", "processed_test_data.csv")
FIGURES_PATH   = os.path.join(BASE_DIR, "experiments", "figures", "eda")

os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(os.path.dirname(TEST_DATA_PATH), exist_ok=True)

# 22-feature map
FEATURE_MAP = {
    "flow_duration"     : "Flow Duration",
    "fwd_iat_mean"      : "Flow IAT Mean",
    "fwd_iat_std"       : "Flow IAT Std",
    "fwd_iat_min"       : "Flow IAT Min",
    "fwd_iat_max"       : "Flow IAT Max",
    "fwd_iat_total"     : "Fwd IAT Total",
    "fwd_pkt_len_mean"  : "Fwd Packet Length Mean",
    "fwd_pkt_len_std"   : "Fwd Packet Length Std",
    "fwd_pkt_len_min"   : "Fwd Packet Length Min",
    "fwd_pkt_len_max"   : "Fwd Packet Length Max",
    "total_fwd_bytes"   : "Total Length of Fwd Packets",
    "flow_pkts_per_sec" : "Flow Packets/s",
    "flow_bytes_per_sec": "Flow Bytes/s",
    "pkt_len_variance"  : "Packet Length Variance",
    "burst_ratio"       : "Fwd IAT Mean",
    "active_time_ratio" : "Active Mean",
    "pkt_count"         : "Total Fwd Packets",
    "unique_ttl_count"  : "Min Packet Length",
    "ttl_mean"          : "Average Packet Size",
    "tcp_flag_ratio"    : "FIN Flag Count",
    "has_udp"           : "URG Flag Count",
    "has_tcp"           : "SYN Flag Count",
}
CIC_FEATURE_COLS = list(FEATURE_MAP.values())
INV_MAP          = {v: k for k, v in FEATURE_MAP.items()}

print("Environment ready.")
print(f"Dataset  : {DATASET_PATH}")
print(f"Figures  : {FIGURES_PATH}")


Environment ready.
Dataset  : /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/datasets/CIC-IDS2017
Figures  : /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda


## 1 · Dataset Ingestion

In [3]:
all_files = sorted(glob.glob(os.path.join(DATASET_PATH, "*.csv")))
print(f"Found {len(all_files)} CSV files:")
for f in all_files:
    print(f"  {os.path.basename(f):<50} {os.path.getsize(f)/1e6:>6.1f} MB")

KEEP_COLS  = CIC_FEATURE_COLS + ["Label", "Destination Port"]
# Destination Port is added so the new "Encrypted Traffic Isolation" section
# (below) can filter to a TLS/HTTPS-only subset — it is not one of the 22
# model features, it is used purely for this EDA-stage subset analysis.
CHUNK_SIZE = 50_000
MAX_ROWS   = 400_000
chunks, total = [], 0

for filepath in all_files:
    if total >= MAX_ROWS:
        break
    try:
        reader = pd.read_csv(
            filepath, chunksize=CHUNK_SIZE, encoding="cp1252",
            low_memory=False,
            usecols=lambda c: c.strip() in KEEP_COLS or c.strip() == "Label",
        )
        for chunk in reader:
            if total >= MAX_ROWS:
                break
            chunk.columns = chunk.columns.str.strip()
            keep  = [c for c in KEEP_COLS if c in chunk.columns]
            chunk = chunk[keep].copy()
            chunk["Label"] = chunk["Label"].astype(str).str.strip().str.upper()
            chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
            chunk.dropna(inplace=True)
            for col in chunk.select_dtypes("float64").columns:
                chunk[col] = chunk[col].astype("float32")
            for col in chunk.select_dtypes("int64").columns:
                chunk[col] = chunk[col].astype("int32")
            chunk["Binary_Label"] = np.where(
                chunk["Label"] == "BENIGN", "Benign", "Malicious")
            chunks.append(chunk)
            total += len(chunk)
        print(f"  {os.path.basename(filepath):<50} cumulative: {total:,}")
    except Exception as exc:
        print(f"  [SKIP] {os.path.basename(filepath)}: {exc}")

df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

assert "Binary_Label" in df.columns
assert df["Binary_Label"].nunique() == 2
print(f"\nLoaded : {len(df):,} rows x {df.shape[1]} columns")
print(f"RAM est.: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
print("\nBinary_Label split:")
print(df["Binary_Label"].value_counts().to_string())
print("\n✓ Dataset verified.")


Found 8 CSV files:
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv     77.1 MB
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv   76.9 MB
  Friday-WorkingHours-Morning.pcap_ISCX.csv            58.3 MB
  Monday-WorkingHours.pcap_ISCX.csv                   176.9 MB
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv   83.1 MB
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv   52.0 MB
  Tuesday-WorkingHours.pcap_ISCX.csv                  135.1 MB
  Wednesday-workingHours.pcap_ISCX.csv                225.2 MB
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   cumulative: 225,711
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv cumulative: 425,457

Loaded : 425,457 rows x 25 columns
RAM est.: 86.6 MB

Binary_Label split:
Binary_Label
Malicious    237293
Benign       188164

✓ Dataset verified.


## 2 · Class Distribution

In [4]:
label_counts  = df["Label"].value_counts()
binary_counts = df["Binary_Label"].value_counts()

fig = plt.figure(figsize=(15, 6), facecolor=BG)
fig.suptitle("CIC-IDS2017  ·  Class Distribution",
             fontsize=14, fontweight="bold", color=BRAND, y=1.01)
gs  = fig.add_gridspec(1, 3, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])

# Horizontal bar (log scale)
bar_colors = [BLUE if l == "BENIGN" else ACCENT for l in label_counts.index]
bars = ax1.barh(label_counts.index, label_counts.values,
                color=bar_colors, edgecolor="white", linewidth=0.7,
                height=0.65, alpha=0.90)
ax1.set_xscale("log")
ax1.set_xlabel("Flow Count  (log scale)", fontsize=9)
ax1.set_title("Attack-Type Breakdown (Log Scale)")
for bar in bars:
    w = bar.get_width()
    ax1.text(w * 1.08, bar.get_y() + bar.get_height() / 2,
             f"{int(w):,}", va="center", ha="left", fontsize=7, color=GREY)
ax1.legend(handles=[
    mpatches.Patch(color=BLUE,   label="Benign"),
    mpatches.Patch(color=ACCENT, label="Malicious"),
], loc="lower right")

# Donut chart
wedge_colors = [PAL[l] for l in binary_counts.index]
wedges, texts, autotexts = ax2.pie(
    binary_counts.values, labels=None, autopct="%1.1f%%",
    colors=wedge_colors, startangle=140, pctdistance=0.72,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight("bold"); at.set_color("white")
ax2.text(0, 0, f"{len(df):,}\nflows", ha="center", va="center",
         fontsize=9, fontweight="bold", color=BRAND)
ax2.set_title("Benign vs Malicious")
ax2.legend(handles=[mpatches.Patch(color=c, label=l)
                    for l, c in zip(binary_counts.index, wedge_colors)],
           loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=2)

save_fig(fig, "class_distribution.png")
plt.show()


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/class_distribution.png


## Encrypted Traffic Isolation — TLS/HTTPS Blind-Spot Analysis

In [5]:
# Encrypted-Traffic Isolation 
# Ports treated as TLS/HTTPS-encrypted for this analysis. 443/8443 = HTTPS;
# 993/995 = IMAPS/POP3S; 465/587 = SMTPS/submission-over-TLS. All are
# encrypted-by-default services, making destination port a defensible proxy
# in the absence of packet-level TLS metadata in this dataset.
ENCRYPTED_PORTS = {443, 8443, 993, 995, 465, 587}

if "Destination Port" not in df.columns:
    raise KeyError(
        "\'Destination Port\' column not found — re-run Section 1 (data loading) "
        "after the KEEP_COLS patch so this column is retained."
    )

df["Destination Port"] = pd.to_numeric(df["Destination Port"], errors="coerce")
encrypted_mask = df["Destination Port"].isin(ENCRYPTED_PORTS)
encrypted_df   = df[encrypted_mask].copy()
plaintext_df   = df[~encrypted_mask].copy()

enc_total = len(encrypted_df)
enc_b     = int((encrypted_df["Binary_Label"] == "Benign").sum())
enc_m     = int((encrypted_df["Binary_Label"] == "Malicious").sum())

print("*" * 100)
print("  ENCRYPTED-TRAFFIC (TLS/HTTPS-CLASS) SUBSET — PORT-BASED ISOLATION")
print("*" * 100)
print(f"  Ports treated as encrypted : {sorted(ENCRYPTED_PORTS)}")
print(f"  Encrypted flows            : {enc_total:,}  ({enc_total/len(df)*100:.1f}% of dataset)")
print(f"  Plaintext / other flows    : {len(plaintext_df):,}")
print(f"  Encrypted — Benign         : {enc_b:,}")
print(f"  Encrypted — Malicious      : {enc_m:,}")
if enc_m > 0:
    print(f"\n  --> Confirms attacks DO occur inside encrypted (port 443-class)")
    print(f"      sessions in this dataset — validating that a DPI-blind,")
    print(f"      flow-based detection approach is necessary for this traffic class,")
    print(f"      not just a theoretical concern.")
else:
    print(f"\n  [NOTE] No malicious flows found at the current port filter.")
    print(f"         Widen ENCRYPTED_PORTS above or inspect the raw Label distribution.")
print("=" * 100)

****************************************************************************************************
  ENCRYPTED-TRAFFIC (TLS/HTTPS-CLASS) SUBSET — PORT-BASED ISOLATION
****************************************************************************************************
  Ports treated as encrypted : [443, 465, 587, 993, 995, 8443]
  Encrypted flows            : 31,549  (7.4% of dataset)
  Plaintext / other flows    : 393,908
  Encrypted — Benign         : 30,810
  Encrypted — Malicious      : 739

  --> Confirms attacks DO occur inside encrypted (port 443-class)
      sessions in this dataset — validating that a DPI-blind,
      flow-based detection approach is necessary for this traffic class,
      not just a theoretical concern.


In [6]:
# Discriminative Power: All Traffic vs Encrypted-Only Traffic
# For each of the 22 behavioral features, compute the KS statistic (Benign vs
# Malicious) twice: once across ALL traffic, once restricted to the
# encrypted-only subset. If KS scores hold up in the encrypted-only column,
# that is direct evidence the features are not merely acting as a proxy for
# "this looks like an encrypted session" — they remain genuinely
# discriminative even when EVERY flow being compared is already encrypted.
available_cols = [c for c in CIC_FEATURE_COLS if c in df.columns]

def ks_per_feature(frame: pd.DataFrame, cols: list) -> pd.Series:
    scores = {}
    b = frame[frame["Binary_Label"] == "Benign"]
    m = frame[frame["Binary_Label"] == "Malicious"]
    for col in cols:
        bv = b[col].dropna()
        mv = m[col].dropna()
        if len(bv) > 10 and len(mv) > 10:
            ks, _ = stats.ks_2samp(bv, mv)
            scores[INV_MAP.get(col, col)] = ks
        else:
            scores[INV_MAP.get(col, col)] = np.nan
    return pd.Series(scores)

ks_all = ks_per_feature(df, available_cols)
ks_enc = ks_per_feature(encrypted_df, available_cols) if enc_m > 5 else pd.Series(dtype=float)

compare_df = pd.DataFrame({"All Traffic": ks_all, "Encrypted-Only": ks_enc}).dropna(how="all")
compare_df = compare_df.sort_values("All Traffic", ascending=True)

fig, ax = plt.subplots(figsize=(11, 8), facecolor=BG)
fig.suptitle(
    "Discriminative Power Survives Encryption\n"
    "KS Statistic — All Traffic vs Encrypted-Only (Port 443-class) Subset",
    fontsize=12, fontweight="bold", color=BRAND
)

y = np.arange(len(compare_df))
h = 0.36
ax.barh(y + h/2, compare_df["All Traffic"].fillna(0),    height=h,
        color="#64B5F6", edgecolor="white", linewidth=0.5, label="All Traffic (mixed)")
ax.barh(y - h/2, compare_df["Encrypted-Only"].fillna(0), height=h,
        color=ACCENT,    edgecolor="white", linewidth=0.5, label="Encrypted-Only (TLS/HTTPS)")

ax.set_yticks(y)
ax.set_yticklabels(compare_df.index, fontsize=8)
ax.axvline(0.3, color=GREY, linestyle="--", linewidth=1, alpha=0.7)
ax.text(0.3, len(compare_df) - 0.5, " weak-signal threshold", fontsize=7, color=GREY, va="top")
ax.set_xlabel("KS statistic  (0 = identical distributions, 1 = fully separated)", fontsize=9)
ax.legend(loc="lower right")

plt.tight_layout()
save_fig(fig, "encrypted_vs_all_ks_comparison.png")
plt.show()

print("\nFeatures that REMAIN discriminative (KS > 0.3) even when restricted to encrypted-only traffic:")
strong_encrypted = compare_df[compare_df["Encrypted-Only"] > 0.3].sort_values("Encrypted-Only", ascending=False)
print(strong_encrypted.to_string() if not strong_encrypted.empty else
      "  (none met threshold at current sample size — see WARN above if encrypted-malicious count is low)")

  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/encrypted_vs_all_ks_comparison.png

Features that REMAIN discriminative (KS > 0.3) even when restricted to encrypted-only traffic:
                    All Traffic  Encrypted-Only
fwd_iat_total          0.171745        0.947322
burst_ratio            0.171745        0.947322
pkt_count              0.175324        0.947322
total_fwd_bytes        0.623101        0.939111
fwd_pkt_len_max        0.724741        0.939111
fwd_pkt_len_mean       0.723125        0.939046
ttl_mean               0.491218        0.934307
flow_duration          0.334882        0.900180
fwd_iat_max            0.328740        0.896610
flow_pkts_per_sec      0.298323        0.895182
fwd_iat_mean           0.195216        0.891903
flow_bytes_per_sec     0.263517        0.844961
fwd_iat_std            0.184031        0.821259
pkt_len_variance       0.367084        0.803213
fwd_pkt_len_std        0.300807        0.796332
fwd_iat_min            0.

In [7]:
# Export: Encrypted-Only Balanced Test Set
# Used by the model validation stage to report precision/recall/F1/ROC-AUC
# SPECIFICALLY on encrypted traffic — the exact claim the proposal makes
# (§1.2, §2.3) that a DPI-blind, flow-based Autoencoder can detect threats
# without payload access. Evaluating only on the general/mixed test set
# (Section 9 below) does not isolate this claim; this file lets the
# validation notebook do so directly.
ENC_TARGET = 1_500
enc_b_pool = encrypted_df[encrypted_df["Binary_Label"] == "Benign"]
enc_m_pool = encrypted_df[encrypted_df["Binary_Label"] == "Malicious"]

if len(enc_m_pool) < 20:
    print(f"[WARN] Only {len(enc_m_pool)} malicious encrypted flows available — "
          f"encrypted-only test set will be small. Consider widening ENCRYPTED_PORTS "
          f"or increasing MAX_ROWS in Section 1.")

enc_b_test = enc_b_pool.sample(n=min(ENC_TARGET, len(enc_b_pool)), random_state=0)
enc_m_test = enc_m_pool.sample(n=min(ENC_TARGET, len(enc_m_pool)), random_state=0)
encrypted_test_df = pd.concat([enc_b_test, enc_m_test], ignore_index=True)

ENCRYPTED_TEST_DATA_PATH = os.path.join(
    BASE_DIR, "experiments", "datasets", "processed_test_data_encrypted.csv")
encrypted_test_df.to_csv(ENCRYPTED_TEST_DATA_PATH, index=False)

print(f"Encrypted-only test set saved : {ENCRYPTED_TEST_DATA_PATH}")
print(f"Shape                         : {encrypted_test_df.shape}")
print(encrypted_test_df["Binary_Label"].value_counts().to_string())

del enc_b_test, enc_m_test, encrypted_test_df, enc_b_pool, enc_m_pool
gc.collect()

Encrypted-only test set saved : /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/datasets/processed_test_data_encrypted.csv
Shape                         : (2239, 25)
Binary_Label
Benign       1500
Malicious     739


2116

## 3 · Behavioral Clustering — IAT Mean vs Packet Length

In [8]:
iat_col = "Flow IAT Mean"
len_col = "Total Length of Fwd Packets"

sample_df = df[[iat_col, len_col, "Binary_Label"]].sample(
    n=min(30_000, len(df)), random_state=42)
sample_df = sample_df[
    (sample_df[iat_col] <= sample_df[iat_col].quantile(0.99)) &
    (sample_df[len_col] <= sample_df[len_col].quantile(0.99))
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle("Behavioral Feature Analysis  ·  IAT Mean vs Fwd Packet Length",
             fontsize=13, fontweight="bold", color=BRAND)

# Scatter
for lbl, grp in sample_df.groupby("Binary_Label"):
    axes[0].scatter(grp[iat_col], grp[len_col],
                    c=PAL[lbl], alpha=0.22, s=5, label=lbl, rasterized=True)
axes[0].set_title("Scatter — IAT vs Packet Volume")
axes[0].set_xlabel("Flow IAT Mean  (s)")
axes[0].set_ylabel("Total Length of Fwd Packets  (bytes)")
axes[0].legend(markerscale=4, title="Class", title_fontsize=8)

# Box plot (no outlier fliers — keeps it clean)
groups = [sample_df[sample_df["Binary_Label"] == lbl][iat_col].values
          for lbl in ["Benign", "Malicious"]]
bp = axes[1].boxplot(groups, patch_artist=True, widths=0.50,
                     showfliers=False,
                     medianprops=dict(color="white", linewidth=2.5),
                     whiskerprops=dict(color=GREY, linewidth=1.2),
                     capprops=dict(color=GREY, linewidth=1.5),
                     boxprops=dict(linewidth=0))
for patch, color in zip(bp["boxes"], [BLUE, ACCENT]):
    patch.set_facecolor(color); patch.set_alpha(0.80)
for i, grp in enumerate(groups, start=1):
    med = np.median(grp)
    axes[1].text(i, med * 1.05, f"med={med:.0f}",
                 ha="center", va="bottom", fontsize=7,
                 color="white", fontweight="bold")
axes[1].set_title("IAT Distribution — Benign vs Malicious")
axes[1].set_xticklabels(["Benign", "Malicious"])
axes[1].set_ylabel("Flow IAT Mean  (s)")

del sample_df; gc.collect()
save_fig(fig, "behavioral_clusters.png")
plt.show()


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/behavioral_clusters.png


## 4 · Statistical Divergence — KDE Plots (KS Test)

In [9]:
KDE_FEATURES = [
    ("Flow Packets/s",              "Flow Packets / s",             (0, 50_000)),
    ("Flow IAT Mean",               "Flow IAT Mean  (s)",           (0, 500_000)),
    ("Fwd Packet Length Mean",      "Fwd Packet Length Mean  (B)",  (0, 1_500)),
    ("Packet Length Variance",      "Packet Length Variance",       (0, 300_000)),
    ("Total Length of Fwd Packets", "Total Fwd Bytes",              (0, 50_000)),
    ("Flow Bytes/s",                "Flow Bytes / s",               (0, 2_000_000)),
]

kde_sample  = df.sample(n=min(50_000, len(df)), random_state=42)
benign_s    = kde_sample[kde_sample["Binary_Label"] == "Benign"]
malicious_s = kde_sample[kde_sample["Binary_Label"] == "Malicious"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor=BG)
fig.suptitle("Statistical Divergence — Benign vs Malicious  (Kolmogorov-Smirnov Test)",
             fontsize=13, fontweight="bold", color=BRAND)
flat_axes = axes.flatten()

for idx, (col, xlabel, xlim) in enumerate(KDE_FEATURES):
    ax = flat_axes[idx]
    ax.set_facecolor(CARD)

    if col not in df.columns:
        ax.set_visible(False)
        continue

    b = benign_s[col].clip(*xlim).dropna()
    m = malicious_s[col].clip(*xlim).dropna()

    if len(b) > 10:
        sns.kdeplot(b, ax=ax, fill=True, alpha=0.50,
                    color=BLUE,  label="Benign",    linewidth=1.5)
    if len(m) > 10:
        sns.kdeplot(m, ax=ax, fill=True, alpha=0.50,
                    color=ACCENT, label="Malicious", linewidth=1.5)

    if len(b) > 0 and len(m) > 0:
        ks, _ = stats.ks_2samp(b, m)
        badge_col = ACCENT if ks > 0.3 else GREY
        ax.text(0.97, 0.95, f"KS = {ks:.3f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=8, fontweight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.3", facecolor=badge_col,
                          edgecolor="none", alpha=0.88))

    ax.set_title(xlabel)
    ax.set_xlim(xlim)
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))
    ax.legend(fontsize=7)

del kde_sample, benign_s, malicious_s; gc.collect()
plt.tight_layout()
save_fig(fig, "statistical_divergence_kde.png")
plt.show()


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/statistical_divergence_kde.png


## 5 · Feature Correlation Matrix — 22-Feature Subset

In [10]:
from matplotlib.colors import LinearSegmentedColormap

available_cols = [c for c in CIC_FEATURE_COLS if c in df.columns]
missing_cols   = [c for c in CIC_FEATURE_COLS if c not in df.columns]
if missing_cols:
    print(f"[WARN] Not in dataset: {[INV_MAP.get(c,c) for c in missing_cols]}")

corr_sample = df[available_cols].sample(n=min(15_000, len(df)), random_state=42)
corr_df     = corr_sample.corr()
corr_df.rename(index=INV_MAP, columns=INV_MAP, inplace=True)
del corr_sample; gc.collect()

cmap_div = LinearSegmentedColormap.from_list(
    "hids_div", ["#1565C0", "#FFFFFF", "#C62828"], N=256)

mask = np.triu(np.ones_like(corr_df, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 11), facecolor=BG)
fig.suptitle("Feature Correlation Matrix — 22 Behavioral Features",
             fontsize=13, fontweight="bold", color=BRAND, y=1.01)

sns.heatmap(
    corr_df, mask=mask, ax=ax,
    cmap=cmap_div, center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 6, "color": "#212121"},
    linewidths=0.35, linecolor=BG,
    square=True,
    cbar_kws={"shrink": 0.75, "label": "Pearson  r", "pad": 0.02},
)
ax.tick_params(axis="x", rotation=45, labelsize=7)
ax.tick_params(axis="y", rotation=0,  labelsize=7)
for spine in ax.spines.values():
    spine.set_visible(True); spine.set_color(GRID); spine.set_linewidth(0.6)

plt.tight_layout()
save_fig(fig, "feature_correlation_matrix.png")
plt.show()


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/feature_correlation_matrix.png


## 6 · Random Forest Feature Importance

In [11]:
rf_cols = [c for c in available_cols if c in df.columns]

N_PER_CLASS = min(3_000, df["Binary_Label"].value_counts().min())
b_samp = df[df["Binary_Label"] == "Benign"   ][rf_cols + ["Binary_Label"]].sample(
             N_PER_CLASS, random_state=42)
m_samp = df[df["Binary_Label"] == "Malicious"][rf_cols + ["Binary_Label"]].sample(
             N_PER_CLASS, random_state=42)
rf_bal = pd.concat([b_samp, m_samp], ignore_index=True).dropna()
del b_samp, m_samp; gc.collect()

X_rf = rf_bal[rf_cols].values
y_rf = (rf_bal["Binary_Label"] == "Malicious").astype(int).values
del rf_bal; gc.collect()

rf = RandomForestClassifier(n_estimators=50, max_depth=10,
                             n_jobs=-1, random_state=42)
rf.fit(X_rf, y_rf)

feat_names = [INV_MAP.get(c, c) for c in rf_cols]
imp        = pd.Series(rf.feature_importances_, index=feat_names).sort_values()
q75        = imp.quantile(0.75)
bar_colors = [ACCENT if v >= q75 else "#64B5F6" for v in imp.values]

fig, ax = plt.subplots(figsize=(13, 6), facecolor=BG)
fig.suptitle("Random Forest Feature Importance — 22-Feature Subset",
             fontsize=13, fontweight="bold", color=BRAND)

bars = ax.barh(imp.index, imp.values,
               color=bar_colors, edgecolor="white", linewidth=0.5,
               height=0.70, alpha=0.92)
ax.axvline(imp.median(), color="orange",  linestyle="--", linewidth=1.5, alpha=0.9)
ax.axvline(q75,          color=ACCENT,    linestyle=":",  linewidth=1.3, alpha=0.8)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.0005, bar.get_y() + bar.get_height() / 2,
            f"{w:.4f}", va="center", fontsize=7, color=GREY)

ax.set_xlabel("Gini Importance", fontsize=9)
ax.set_ylabel("Feature", fontsize=9)
ax.legend(handles=[
    mpatches.Patch(color=ACCENT,    label="Top-quartile features"),
    mpatches.Patch(color="#64B5F6", label="Lower-importance features"),
    plt.Line2D([0],[0], color="orange", linestyle="--", label="Median"),
    plt.Line2D([0],[0], color=ACCENT,   linestyle=":",  label="75th percentile"),
], fontsize=8)

plt.tight_layout()
save_fig(fig, "rf_feature_importance.png")
plt.show()
print("\nTop-5 features:")
print(imp.tail(5).sort_values(ascending=False).to_string())


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/rf_feature_importance.png

Top-5 features:
total_fwd_bytes     0.154736
fwd_pkt_len_max     0.141409
ttl_mean            0.121822
fwd_pkt_len_mean    0.093178
pkt_len_variance    0.073553


## 7 · PCA 2D Projection — Feature Separability

In [12]:
from matplotlib.patches import Ellipse

N_PCA = min(3_000, df["Binary_Label"].value_counts().min())
b_pca = df[df["Binary_Label"] == "Benign"   ][rf_cols].sample(N_PCA, random_state=42)
m_pca = df[df["Binary_Label"] == "Malicious"][rf_cols].sample(N_PCA, random_state=42)
labels_pca = np.array(["Benign"] * N_PCA + ["Malicious"] * N_PCA)

X_raw = pd.concat([b_pca, m_pca], ignore_index=True).dropna().values
labels_pca = labels_pca[:len(X_raw)]
del b_pca, m_pca; gc.collect()

X_sc  = MinMaxScaler().fit_transform(X_raw)
pca   = PCA(n_components=2, random_state=42)
X_2d  = pca.fit_transform(X_sc)
ev    = pca.explained_variance_ratio_
del X_raw, X_sc; gc.collect()

fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG)
fig.suptitle("PCA 2D Projection — 22 Behavioral Features",
             fontsize=13, fontweight="bold", color=BRAND)

for lbl in ["Benign", "Malicious"]:
    mask_pca = labels_pca == lbl
    ax.scatter(X_2d[mask_pca, 0], X_2d[mask_pca, 1],
               c=PAL[lbl], alpha=0.30, s=9,
               label=f"{lbl}  ({mask_pca.sum():,})", rasterized=True)

# Confidence ellipses (1-sigma and 2-sigma) — visual density cue
for lbl, color in PAL.items():
    pts = X_2d[labels_pca == lbl]
    cx, cy = pts[:,0].mean(), pts[:,1].mean()
    sx, sy = pts[:,0].std(),  pts[:,1].std()
    for n_std, alpha in [(1, 0.20), (2, 0.09)]:
        ax.add_patch(Ellipse((cx, cy), 2*n_std*sx, 2*n_std*sy,
                             linewidth=1.2, edgecolor=color,
                             facecolor=color, alpha=alpha))

ax.set_xlabel(f"PC1  ({ev[0]*100:.1f} % variance)", fontsize=9)
ax.set_ylabel(f"PC2  ({ev[1]*100:.1f} % variance)", fontsize=9)
ax.set_title(f"Total variance explained: {sum(ev)*100:.1f} %", fontsize=9)
ax.legend(markerscale=3, title="Class", title_fontsize=8)

plt.tight_layout()
save_fig(fig, "pca_feature_projection.png")
plt.show()


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/pca_feature_projection.png


## 8 · Attack-Type Feature Profiles vs Benign Baseline

In [13]:
PROFILE_COLS  = ["Flow Packets/s", "Flow IAT Mean", "Total Length of Fwd Packets"]
avail_profile = [c for c in PROFILE_COLS if c in df.columns]

top_attacks  = (df[df["Label"] != "BENIGN"]["Label"]
                .value_counts().head(6).index.tolist())
profile_mask = df["Label"].isin(["BENIGN"] + top_attacks)
profile_df   = df.loc[profile_mask, avail_profile + ["Label"]].copy()

# Clean, readable short labels
label_rename = {
    "BENIGN": "Benign", "DDOS": "DDoS",
    "DOS HULK": "DoS Hulk", "DOS GOLDENEYE": "DoS GoldenEye",
    "DOS SLOWLORIS": "DoS Slowloris", "FTP-PATATOR": "FTP Patator",
    "SSH-PATATOR": "SSH Patator", "BOT": "Bot",
    "INFILTRATION": "Infiltration", "PORTSCAN": "Port Scan",
    "HEARTBLEED": "Heartbleed",
    "WEB ATTACK - BRUTE FORCE": "Brute Force",
    "WEB ATTACK - XSS": "XSS",
    "WEB ATTACK - SQL INJECTION": "SQL Injection",
}
profile_df["Label"] = profile_df["Label"].replace(label_rename)

fig, axes = plt.subplots(1, len(avail_profile), figsize=(14, 5), facecolor=BG)
fig.suptitle("Attack-Type Feature Profiles vs Benign Baseline",
             fontsize=13, fontweight="bold", color=BRAND)
if len(avail_profile) == 1:
    axes = [axes]

for ax, col in zip(axes, avail_profile):
    p95   = profile_df[col].quantile(0.95)
    means = (profile_df[profile_df[col] <= p95]
             .groupby("Label")[col].mean()
             .sort_values())
    bar_col = ["#1E88E5" if l == "Benign" else ACCENT for l in means.index]
    bars = ax.barh(means.index, means.values,
                   color=bar_col, edgecolor="white", linewidth=0.5,
                   height=0.65, alpha=0.88)
    ax.axvline(0, color=GREY, linewidth=0.8)
    # Hatch benign bar for clear identification
    for bar, lbl in zip(bars, means.index):
        if lbl == "Benign":
            bar.set_hatch("///"); bar.set_edgecolor("#1565C0"); bar.set_linewidth(1.2)
    ax.set_title(col.replace("/s", " / s"), fontsize=9)
    ax.set_xlabel("Mean  (clipped @ p95)", fontsize=8)

del profile_df; gc.collect()
plt.tight_layout()
save_fig(fig, "attack_type_profiles.png")
plt.show()


  Saved → /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/figures/eda/attack_type_profiles.png


## 9 · Test Set Export

In [14]:
TARGET_COUNT = 3_000
b_test = df[df["Binary_Label"] == "Benign"   ].sample(
             n=min(TARGET_COUNT, (df["Binary_Label"]=="Benign").sum()),    random_state=0)
m_test = df[df["Binary_Label"] == "Malicious"].sample(
             n=min(TARGET_COUNT, (df["Binary_Label"]=="Malicious").sum()), random_state=0)
test_df = pd.concat([b_test, m_test], ignore_index=True)
del b_test, m_test; gc.collect()

test_df.to_csv(TEST_DATA_PATH, index=False)
print(f"Test set saved : {TEST_DATA_PATH}")
print(f"Shape          : {test_df.shape}")
print("\nClass split:")
print(test_df["Binary_Label"].value_counts().to_string())
del test_df; gc.collect()


Test set saved : /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/datasets/processed_test_data.csv
Shape          : (6000, 25)

Class split:
Binary_Label
Benign       3000
Malicious    3000


0

## 10 · EDA Summary

In [15]:
benign_n    = int((df["Binary_Label"] == "Benign").sum())
malicious_n = int((df["Binary_Label"] == "Malicious").sum())

print("*" * 100)
print("  AUTOENCODER HIDS  -  EDA SUMMARY")
print("*" * 100)
print(f"  Dataset       : CIC-IDS2017")
print(f"  CSV files     : {len(all_files)}")
print(f"  Total rows    : {len(df):,}  (capped at {MAX_ROWS:,})")
print(f"  RAM usage     : {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"  Benign flows  : {benign_n:,}")
print(f"  Malicious     : {malicious_n:,}")
print(f"  Imbalance     : {benign_n/max(malicious_n,1):.1f}x more benign")
print("-" * 100)
print(f"  Encrypted (port 443-class) flows : {enc_total:,}  ({enc_total/len(df)*100:.1f}% of dataset)")
print(f"  Encrypted — Benign                : {enc_b:,}")
print(f"  Encrypted — Malicious             : {enc_m:,}")
print(f"  Encrypted-only test set           : {ENCRYPTED_TEST_DATA_PATH}")
print("-" * 100)
print(f"  Feature cols found  : {len(available_cols)} / {len(CIC_FEATURE_COLS)}")
print(f"  Missing cols        : {[INV_MAP.get(c,c) for c in missing_cols]}")
print("-" * 100)
print(f"  Figures saved to    : {FIGURES_PATH}/")
for fname in sorted(os.listdir(FIGURES_PATH)):
    if fname.endswith(".png"):
        size_kb = os.path.getsize(os.path.join(FIGURES_PATH, fname)) / 1024
        print(f"    [OK] {fname:<45} {size_kb:>6.1f} KB")



****************************************************************************************************
  AUTOENCODER HIDS  -  EDA SUMMARY
****************************************************************************************************
  Dataset       : CIC-IDS2017
  CSV files     : 8
  Total rows    : 425,457  (capped at 400,000)
  RAM usage     : 86.6 MB
  Benign flows  : 188,164
  Malicious     : 237,293
  Imbalance     : 0.8x more benign
----------------------------------------------------------------------------------------------------
  Encrypted (port 443-class) flows : 31,549  (7.4% of dataset)
  Encrypted — Benign                : 30,810
  Encrypted — Malicious             : 739
  Encrypted-only test set           : /home/l391lninj1/Desktop/AUTOENCODER_HIDS/experiments/datasets/processed_test_data_encrypted.csv
----------------------------------------------------------------------------------------------------
  Feature cols found  : 22 / 22
  Missing cols        : []
-------